In [1]:
import numpy as np
import pandas as pd
import requests

In [2]:
# IMPORTING The COUNTRIES
url = "https://api.worldbank.org/countries?format=json&per_page=300"

response = requests.get(url)
response.status_code

200

In [3]:
data = response.json()
print(data[0])
print(len(data))

#1 - METADATA
#2 - ACTUAL DATA

{'page': 1, 'pages': 1, 'per_page': '300', 'total': 296}
2


In [4]:
countries = data[1]
countries = pd.DataFrame(countries)

In [5]:
countries["incomeLevel"][6]

{'id': 'UMC', 'iso2code': 'XT', 'value': 'Upper middle income'}

In [6]:
countries["region"]= countries["region"].apply(lambda x:x["value"])

In [7]:
countries["incomeLevel"]= countries["incomeLevel"].apply(lambda x:x["value"])

In [8]:
countries["lendingType"]= countries["lendingType"].apply(lambda x:x["value"])

In [9]:
countries.drop(columns= ["adminregion", "adminregion"], inplace= True)

In [10]:
countries.drop(columns= ["capitalCity"], inplace= True)

In [11]:
countries["region"].unique()

<ArrowStringArray>
[                       'Latin America & Caribbean ',
                                        'Aggregates',
 'Middle East, North Africa, Afghanistan & Pakistan',
                               'Sub-Saharan Africa ',
                             'Europe & Central Asia',
                               'East Asia & Pacific',
                                        'South Asia',
                                     'North America']
Length: 8, dtype: str

In [12]:
countries.rename(columns={"iso2Code":"country_id"}, inplace= True)

In [13]:
countries[countries["region"]== "Aggregates"]

,id,country_id,name,region,incomeLevel,lendingType,longitude,latitude
1,AFE,ZH,Africa Eastern and Southern,Aggregates,Aggregates,Aggregates,,
3,AFR,A9,Africa,Aggregates,Aggregates,Aggregates,,
4,AFW,ZI,Africa Western and Central,Aggregates,Aggregates,Aggregates,,
8,ARB,1A,Arab World,Aggregates,Aggregates,Aggregates,,
18,BEA,B4,East Asia & Pacific (IBRD-only countries),Aggregates,Aggregates,Aggregates,,
...,...,...,...,...,...,...,...,...
269,TSA,T5,South Asia (IDA & IBRD),Aggregates,Aggregates,Aggregates,,
270,TSS,T6,Sub-Saharan Africa (IDA & IBRD countries),Aggregates,Aggregates,Aggregates,,
278,UMC,XT,Upper middle income,Aggregates,Aggregates,Aggregates,,
288,WLD,1W,World,Aggregates,Aggregates,Aggregates,,


In [14]:
# CODE For INDICATOR's
base_url = "https://api.worldbank.org/v2/indicators?format=json"
response = requests.get(base_url)
response.status_code

200

In [15]:
indicators_data = response.json()

In [16]:
indicators_data[0]

{'page': 1, 'pages': 591, 'per_page': '50', 'total': 29544}

In [17]:
indicators_data[1]

[{'id': '1.0.HCount.1.90usd',
  'name': 'Poverty Headcount ($1.90 a day)',
  'unit': '',
  'source': {'id': '37', 'value': 'LAC Equity Lab'},
  'sourceNote': 'The poverty headcount index measures the proportion of the population with daily per capita income (in 2011 PPP) below the poverty line.',
  'sourceOrganization': 'LAC Equity Lab tabulations of SEDLAC (CEDLAS and the World Bank).',
  'topics': [{'id': '11', 'value': 'Poverty '}]},
 {'id': '1.0.HCount.2.5usd',
  'name': 'Poverty Headcount ($2.50 a day)',
  'unit': '',
  'source': {'id': '37', 'value': 'LAC Equity Lab'},
  'sourceNote': 'The poverty headcount index measures the proportion of the population with daily per capita income (in 2005 PPP) below the poverty line.',
  'sourceOrganization': 'LAC Equity Lab tabulations of SEDLAC (CEDLAS and the World Bank).',
  'topics': [{'id': '11', 'value': 'Poverty '}]},
 {'id': '1.0.HCount.Mid10to50',
  'name': 'Middle Class ($10-50 a day) Headcount',
  'unit': '',
  'source': {'id': '37

In [18]:
all_dfs = []
for  i in range(1,526):
  url= f"https://api.worldbank.org/v2/indicators?format=json&per_page=500&page={i}"
  response= requests.get(url)

  if response.status_code == 200:
    data=response.json()

    if len(data) < 2:
      print(f"No data at page{i}")

    indicators=data[1]
    df= pd.DataFrame([{"id":item["id"],
                   "name":item["name"]} for item in indicators ])
    all_dfs.append(df)
    print(f"Page{i}: {len(df)} indicators collected")
  else:
    print(f"Failed to fetch  page {i},status code {response.status_code}")

Page1: 500 indicators collected
Page2: 500 indicators collected
Page3: 500 indicators collected
Page4: 500 indicators collected
Page5: 500 indicators collected
Page6: 500 indicators collected
Page7: 500 indicators collected
Page8: 500 indicators collected
Page9: 500 indicators collected
Page10: 500 indicators collected
Page11: 500 indicators collected
Page12: 500 indicators collected
Page13: 500 indicators collected
Page14: 500 indicators collected
Page15: 500 indicators collected
Page16: 500 indicators collected
Page17: 500 indicators collected
Page18: 500 indicators collected
Page19: 500 indicators collected
Page20: 500 indicators collected
Page21: 500 indicators collected
Page22: 500 indicators collected
Page23: 500 indicators collected
Page24: 500 indicators collected
Page25: 500 indicators collected
Page26: 500 indicators collected
Page27: 500 indicators collected
Page28: 500 indicators collected
Page29: 500 indicators collected
Page30: 500 indicators collected
Page31: 500 indicat

KeyboardInterrupt: 

In [19]:
final_df = pd.concat(all_dfs, ignore_index= True)

In [20]:
final_df.to_csv("final_df.csv")

In [21]:
# EXTRACT VALUE For VARIOUD INDICATOR's UNDER The DOMAIN For EACH COUNTRY
indicator_groups = {"economic_activity_growth": ["NY.GDP.MKTP.KD.ZG",   # GDP GROWTH (Annual %)
                                                 "NY.GDP.PCAP.CD"],     # GDP PER CAPITA (Current US$)
                                                 
                    "labour_market_indicators": ["SL.UEM.TOTL.ZS",      # TOTAL UNEMPLOYMENT
                                                 "SL.UEM.1524.ZS",      # TOTAL YOUTH UNEMPLOYMENT (Age's 15-24)
                                                 "SL.TLF.TOTL.IN"],     # TOTAL LABOUR FORCE],

                    "trade_globalization": ["NE.EXP.GNFS.CD",           # EXPORT's Of GOOD's & SERVICE's (Current $)
                                            "NE.IMP.GNFS.CD"],          # IMPORT's Of GOOD's & SERVICE's (Current US$)
                    
                    "poverty_inequality": ["SI.POV.NAHC",               # POVERTY HEADCOUNT RATIO At NATIONAL POVERTY LINE's (% Of Population)
                                           "SI.POV.GINI"],              # GINI INDEX (Measure Of Income Inequality)
                    
                    "environmental_indicators": ["EG.FEC.RNEW.ZS",      # RENEWABLE ENERGY CONSUMPTION (% Of Total Final Energy Consumption)
                                                 "AG.LND.FRST.ZS"],     # FOREST AREA (% Of Land Area)
                    
                    "health_indicators": ["SP.DYN.LE00.IN",             # LIFE EXPECTANCY At BIRTH
                                          "SP.DYN.IMRT.IN",             # INFANT MORTALITY RATE
                                          "SH.H2O.BASW.ZS",             # ACCESS to At Least BASIC WATER SERVICE's (% Of Population)
                                          "SH.XPD.CHEX.GD.ZS",          # CURRENT HEALTH EXPENDITURE (% Of Gdp)
                                          "SH.IMM.IDPT",                # IMMUNIZATION, DPT (% Of Children Age's 12-23 Month's)
                                          "SH.IMM.MEAS",                # IMMUNIZATION, MEASLES (% Of Children Age's 12-23 Month's)
                                          "SH.MMR.RISK.ZS",             # RISK Of MATERNAL DEATH
                                          "SH.DTH.COMM.ZS",             # DEATH's From COMMUNICABLE DISEASE's (% Of Total)
                                          "SH.TBS.INCD",                # TUBERCULOSIS INCIDENCE (Per 100,000 People)
                                          "SH.STA.BRTC.ZS",             # BIRTH's ATTENTED By SKILLED HEALTH STAFF (%)
                                          "SH.STA.MMRT",                # MATERNAL MORTALITY RATIO (Modeled Estimate, Per 100,00 Live Birth's)
                                          "SP.POP.65UP.TO.ZS",          # POPULATION AGE's 65 & ABOVE (% Of Total Population)
                                          "SH.HIV.INCD.ZS"],            # HIV INCIDENCE RATE (Per 1,000 Uninfected Population Age's 15-49)
                    
                    "technology_indicators": ["IT.NET.USER.ZS",         # INDIVIDUAL's USING The INTERNET (% Of Population)
                                              "IT.CEL.SETS.P2"]}        # MOBILE CELLULAR SUBSCRIPTION's (Per 100 People)

In [22]:
for i, j in indicator_groups.items():
  print(j)

['NY.GDP.MKTP.KD.ZG', 'NY.GDP.PCAP.CD']
['SL.UEM.TOTL.ZS', 'SL.UEM.1524.ZS', 'SL.TLF.TOTL.IN']
['NE.EXP.GNFS.CD', 'NE.IMP.GNFS.CD']
['SI.POV.NAHC', 'SI.POV.GINI']
['EG.FEC.RNEW.ZS', 'AG.LND.FRST.ZS']
['SP.DYN.LE00.IN', 'SP.DYN.IMRT.IN', 'SH.H2O.BASW.ZS', 'SH.XPD.CHEX.GD.ZS', 'SH.IMM.IDPT', 'SH.IMM.MEAS', 'SH.MMR.RISK.ZS', 'SH.DTH.COMM.ZS', 'SH.TBS.INCD', 'SH.STA.BRTC.ZS', 'SH.STA.MMRT', 'SP.POP.65UP.TO.ZS', 'SH.HIV.INCD.ZS']
['IT.NET.USER.ZS', 'IT.CEL.SETS.P2']


In [ ]:
import time
base_url = "https://api.worldbank.org/countries/all/indicators/{}?format=json&per_page=1000&page={}"

category_dataframes = {}
for category,indicators in indicator_groups.items():
  print(f"Fetch Information for Category: {category}")
  all_dfs_for_category=[]

  for indicator_code in indicators:
    print(f"Fetching Indicator: {indicator_code}")
    page=1

    while True:
      url= base_url.format(indicator_code, page)
      reponse= requests.get(url)
      if response.status_code!=200:
        print(f"No Data for Indicator{indicator_code} on Page {page}")
        break

      data= reponse.json()
      if len(data)<2:
        print(f"Failed at Page{page}")
        break

      total_pages= data[0]["pages"]
      record= data[1]

      df= pd.json_normalize(record)
      df= df[["country.id", "country.value", "indicator.id",
              "indicator.value", "date", "value"]].rename(columns= {"country.id": "country_id",
                                                                 "country.value": "country_value",
                                                                 "indicator.id": "indicator_id",
                                                                 "indicator.value": "indicator_name",
                                                                 "date": "year"})

      df= df[df["year"].astype(int)>2015]
      all_dfs_for_category.append(df)

      if page >= total_pages:
        break
      else:
        page += 1
        time.sleep(0.3)

  if all_dfs_for_category:
      combined_df= pd.concat(all_dfs_for_category, ignore_index= True)
      category_dataframes[category]= combined_df
      print(f"Total Rows Collected for {category}:{len(combined_df)}")
  else:
      print(f"No Data Collected for {category}")

print("Data Fetching Completed")

In [25]:
economic_activity = category_dataframes.get("economic_activity_growth", pd.DataFrame())
labour_market_jobs = category_dataframes.get("labour_market_indicators", pd.DataFrame())
trade_globalization = category_dataframes.get("trade_globalization", pd.DataFrame())
poverty_inequality = category_dataframes.get("poverty_inequality", pd.DataFrame())
environmental_indicators = category_dataframes.get("environmental_indicators", pd.DataFrame())
health_indicators = category_dataframes.get("health_indicators", pd.DataFrame())
technology_indicators = category_dataframes.get("technology_indicators", pd.DataFrame())

In [ ]:
economic = pd.merge(economic_activity, countries, on = "country_id", how = "inner")
labour_market = pd.merge(labour_market_jobs, countries, on = "country_id", how = "inner")
trade = pd.merge(trade_globalization, countries, on = "country_id", how = "inner")
poverty = pd.merge(poverty_inequality, countries, on = "country_id", how = "inner")
environment = pd.merge(environmental_indicators, countries, on = "country_id", how = "inner")
health = pd.merge(health_indicators, countries, on = "country_id", how = "inner")
technology = pd.merge(technology_indicators, countries, on = "country_id", how = "inner")

In [ ]:
economic.drop(columns= ["indicator_id", "name", "id"], inplace= True)
trade.drop(columns= ["indicator_id", "name", "id"], inplace= True)
labour_market.drop(columns= ["indicator_id", "name", "id"], inplace= True)
poverty.drop(columns= ["indicator_id", "name", "id"], inplace= True)
environment.drop(columns= ["indicator_id", "name", "id"], inplace= True)
health.drop(columns= ["indicator_id", "name", "id"], inplace= True)
technology.drop(columns= ["indicator_id", "name", "id"], inplace= True)

In [ ]:
health

In [ ]:
df_wide = health.pivot_table(index= ["country_value", "year"],
                             columns= "indicator_name",
                             values= "value")

In [ ]:
df_wide = health.pivot_table(index= ["country_value", "year"],
                             columns= "indicator_name",
                             values= "value")
corr = df_wide.corr()

import seaborn as sns
import matplotlib.pyplot as plt

plt.figure(figsize= (16, 16), facecolor= "none")
ax= sns.heatmap(corr,
                annot= True,
                cmap= "coolwarm",
                cbar_kws= {"label": "Correlation"},
                annot_kws= {"fontsize": 14, "fontweight": "bold"})
plt.title("RELATIONSHIP BETWEEN VARIOUS HEALTH INDICATORS")

ax.tick_params(axis= "x", labelsize= 14)
ax.tick_params(axis= "y", labelsize= 14)
plt.show()

In [ ]:
df_pivot = health.pivot(index= ["country_value", "year"], columns= "indicator_name", values= "value").reset_index()

plt.figure(figsize=(12, 8), facecolor= "none")

ax=sns.regplot(data= df_pivot,
               x= "Current Health Expenditure (% Of Gdp)",
               y= "Total, Life Expectancy At Birth (Year's)",
               scatter_kws= {"alpha":0.5, "s":40},
               line_kws= {"color":"red", "lw":2})

plt.title("RELATIONSHIP BETWEEN EXPENDITURE & LIFE EXPECTANCY")
plt.xlabel("Current Health Expenditure (% Of Gdp)")
plt.ylabel("Life Expectancy At Birth")
plt.grid(True)
plt.show()